In [1]:
import torch
import torch.nn as nn
import math

def create_causal_mask(sequence_length: int) -> torch.Tensor:
    # Upper triangular positions above the diagonal are True.
    future_positions = torch.triu(
        torch.ones(sequence_length, sequence_length, dtype=torch.bool),
        diagonal=1,
    )
    # print(future_positions)

    # Future positions become -inf; current and past positions become 0.
    mask = torch.zeros(sequence_length, sequence_length)
    # print(mask)
    mask = mask.masked_fill(future_positions, float("-inf"))

    return mask


def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    attention_dropout: float | None = None,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    d_k = Q.shape[-1] // Q.shape[1]
    # Q, K, V: [B, num_heads, L, d_k]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Masked positions receive zero probability after softmax.
    if mask is not None:
        scores = scores + mask

    attention_probs = scores.softmax(dim=-1)
    # print(attention_probs)

    if attention_dropout is not None:
        attention_probs = attention_dropout(attention_probs)

    # [B, num_heads, L, d_k]
    output = torch.matmul(attention_probs, V)

    return output

In [2]:
mask = create_causal_mask(4)
x = torch.randn(4, 4, 4, 8)
out = scaled_dot_product_attention(x, x, x, mask=mask)
print(out.shape)
tokens = torch.zeros(4, 77, dtype=torch.long)

tokens[:, 0] = 49406  # Start-of-text token
tokens[:, 1] = torch.randint(1, 1000, (4,))
tokens[:, 2] = 49407  # End-of-text token
# Remaining positions are padding.


logit_scale = nn.Parameter(
    torch.ones(()) * torch.log(torch.tensor(1 / 0.07))
)

print("Log scale:", logit_scale.item())
print("Actual scale:", logit_scale.exp().item())

assert torch.isclose(
    logit_scale.exp(),
    torch.tensor(1 / 0.07),
    atol=1e-5,
)

torch.Size([4, 4, 4, 8])
Log scale: 2.6592600345611572
Actual scale: 14.285714149475098


In [4]:
def test_tokenizer():
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    tokens = tok(
        ["a photo of a cat", "a photo of a dog"],
        padding="max_length", max_length=77, truncation=True, return_tensors="pt",
    )["input_ids"]      # [2, 77]
    print(tokens)

In [5]:
test_tokenizer()

/home/etman/etman/paper-implementations/CLIP/.venv/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[49406,   320,  1125,   539,   320,  2368, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407],
        [49406,   320,  1125,   539,   320,  1929, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407,
         49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 49407, 

In [ ]:
# import logging

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
#     filename="app.log",
#     filemode="a",
# )

# log = logging.getLogger(__name__)

# log.info("Application started")
# log.warning("This is a warning")
# log.error("Something went wrong")

In [2]:
from src.model import CLIP
model = CLIP()

In [4]:
print(type(model.named_parameters()))

<class 'generator'>


In [17]:
import torch
import torch.nn as nn

def get_params_group(model, weight_decay):
    decay_params = []
    no_decay_params = []
    
    for name, param in model.named_parameters():
        # print(f"name: {name}")
        # print(f"param: {param.shape}")
        # print("__________________\n")
        if not param.requires_grad:
            continue
        if param.ndim < 2:
            no_decay_params.append(param) # biases, LayerNorm gains and biases, logit_scale
        else:
            decay_params.append(param) # linear / conv / embedding weights, cls token, position
            
    return [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]
        

In [18]:
out = get_params_group(model=model, weight_decay=0.2)

In [19]:
out

[{'params': [Parameter containing:
   tensor([[[[-3.5543e-02, -7.1834e-03,  1.8560e-02,  ..., -2.9739e-02,
              -2.1527e-02, -1.5727e-02],
             [ 1.7717e-02, -3.1931e-02, -1.3439e-03,  ..., -5.1916e-03,
               3.0372e-02, -1.8323e-02],
             [ 1.4017e-02, -2.3647e-02, -2.9395e-02,  ..., -1.9271e-02,
              -2.1207e-02,  4.2083e-05],
             ...,
             [-1.1880e-03, -2.5351e-02, -3.4990e-02,  ...,  9.6221e-04,
               3.2510e-02, -2.5229e-02],
             [-1.2718e-02,  2.0986e-02, -2.0275e-02,  ...,  1.1395e-02,
              -2.1862e-02, -2.0001e-02],
             [-3.2204e-02,  3.4875e-02, -4.3322e-03,  ..., -2.4674e-02,
               1.1884e-02,  9.0451e-03]],
   
            [[-1.7958e-02,  2.5746e-02,  2.9140e-02,  ...,  1.9837e-02,
               2.9642e-02,  9.5886e-03],
             [-2.5870e-02,  4.0244e-03, -3.5291e-02,  ..., -4.1786e-03,
               2.4196e-02, -2.9770e-03],
             [ 1.1711e-02,  2.6771e-02

In [ ]:
import torch
import torch.nn as nn


def create_optimizer(
    model: nn.Module,
    learning_rate: float = 1e-4,
    weight_decay: float = 0.2,
) -> torch.optim.AdamW:
    """Create AdamW with no weight decay on biases or normalization gains."""

    decay_params = []
    no_decay_params = []

    norm_types = (
        nn.LayerNorm,
        nn.BatchNorm1d,
        nn.BatchNorm2d,
        nn.GroupNorm,
    )

    for module in model.modules():
        for name, param in module.named_parameters(recurse=False):
            if not param.requires_grad:
                continue

            if name == "bias" or (
                name == "weight" and isinstance(module, norm_types)
            ):
                no_decay_params.append(param)
            else:
                decay_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": decay_params,
                "weight_decay": weight_decay,
            },
            {
                "params": no_decay_params,
                "weight_decay": 0.0,
            },
        ],
        lr=learning_rate,
        betas=(0.9, 0.98),
        eps=1e-6,
    )

    return optimizer